# PACE IOP data

In [1]:
# imports
from importlib import reload
import os
import numpy as np

from matplotlib import pyplot as plt

import xarray
import pandas

from ocpy.pace import io as pace_io
from ocpy.utils import plotting
from ocpy.utils import coords as ocpy_coords
from ocpy.water import scattering

from bing.parameters import standard
from bing.models import utils as model_utils
from bing.priors import priors as bing_priors
from bing.fitting import inference as bing_inf
from bing import evaluate
from bing.fitting import chisq_fit
from bing import plotting as bing_plotting

# Locals
from grab_pace_granules import load_from_json
import fitting as m_fitting
import grab_pace_granules

# Load up

In [2]:
match_file = 'matched_argo_bgc_profiles_bbp.csv'
# Load up Argo profiles, already matched to PACE
matched = pandas.read_csv(match_file)

# Load up PACE granules
granules, pace = load_from_json('PACE_50clouds.json')


In [3]:
matched.pace_ids

0      PACE_OCI_L2_AOP_PACE_OCI.20240708T202028.L2.OC...
1      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
2      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
3      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
4      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
                             ...                        
807    PACE_OCI_L2_AOP_PACE_OCI.20250327T222202.L2.OC...
808    PACE_OCI_L2_AOP_PACE_OCI.20250209T163639.L2.OC...
809    PACE_OCI_L2_AOP_PACE_OCI.20250219T155847.L2.OC...
810    PACE_OCI_L2_AOP_PACE_OCI.20250301T152050.L2.OC...
811    PACE_OCI_L2_AOP_PACE_OCI.20250330T160134.L2.OC...
Name: pace_ids, Length: 812, dtype: object

In [4]:
pace

,id,polygon,time,CC,url
0,PACE_OCI_L2_AOP_PACE_OCI.20240305T045359.L2.OC...,"POLYGON ((129.99634 19.95012, 105.26324 14.805...",2024-03-05 04:56:28.500000+00:00,49.5,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
1,PACE_OCI_L2_AOP_PACE_OCI.20240305T081040.L2.OC...,"POLYGON ((80.81132 19.9969, 56.07363 14.85543,...",2024-03-05 08:13:10+00:00,48.2,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
2,PACE_OCI_L2_AOP_PACE_OCI.20240305T112721.L2.OC...,"POLYGON ((31.63485 20.04028, 6.9071 14.89843, ...",2024-03-05 11:29:50.500000+00:00,42.7,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
3,PACE_OCI_L2_AOP_PACE_OCI.20240305T143902.L2.OC...,"POLYGON ((-14.32906 2.23116, -38.0179 -2.80614...",2024-03-05 14:41:31.500000+00:00,46.1,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
4,PACE_OCI_L2_AOP_PACE_OCI.20240305T162722.L2.OC...,"POLYGON ((-44.22893 37.98507, -73.20055 32.449...",2024-03-05 16:29:51.500000+00:00,48.9,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
...,...,...,...,...,...
4649,PACE_OCI_L2_AOP_PACE_OCI.20250501T163649.L2.OC...,"POLYGON ((-39.70046 -17.20359, -64.99458 -22.4...",2025-05-01 16:39:18.500000+00:00,34.9,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
4650,PACE_OCI_L2_AOP_PACE_OCI.20250501T165149.L2.OC...,"POLYGON ((-50.29488 41.55137, -80.64542 35.865...",2025-05-01 16:54:18.500000+00:00,47.6,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
4651,PACE_OCI_L2_AOP_PACE_OCI.20250501T195832.L2.OC...,"POLYGON ((-93.50951 0.57069, -117.21605 -4.479...",2025-05-01 20:01:01.500000+00:00,46.2,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...
4652,PACE_OCI_L2_AOP_PACE_OCI.20250501T201332.L2.OC...,"POLYGON ((-99.51873 59.34401, -141.90399 52.34...",2025-05-01 20:16:01.500000+00:00,44.9,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...


In [6]:
iop_url = pace.iloc[0].url.replace('AOP', 'IOP')
iop_url

'https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240305T045359.L2.OC_IOP.V3_0.nc'

## Used wget

## Load it

In [8]:
iop_file = os.path.join(os.getenv('OS_COLOR'), 'PACE', 'L2_IOP', 
                     os.path.basename(iop_url))
iop_file

'/home/xavier/Projects/Oceanography/data/Color/PACE/L2_IOP/PACE_OCI.20240305T045359.L2.OC_IOP.V3_0.nc'

In [15]:
reload(pace_io)
xds_iop = pace_io.load_iop_l2(iop_file)

## AOP too

In [21]:
aop_file = '/home/xavier/Projects/Oceanography/data/Color/PACE/L2_AOP/PACE_OCI.20240510T203409.L2.OC_AOP.V3_0.nc' 

In [ ]:
reload(pace_io)
xds_iop = pace_io.load_oci_l2(aop_file)

Python 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]
Type 'copyright', 'credits' or 'license' for more information
IPython 8.29.0 -- An enhanced Interactive Python. Type '?' for help.


116 of io.py



In [1]:  gd.variables


Out[1]: 
{'Rrs': <class 'netCDF4.Variable'>
 int16 Rrs(number_of_lines, pixels_per_line, wavelength_3d)
     long_name: Remote sensing reflectance
     scale_factor: 2e-06
     add_offset: 0.05
     units: sr^-1
     coordinates: longitude latitude
     standard_name: surface_ratio_of_upwelling_radiance_emerging_from_sea_water_to_downwelling_radiative_flux_in_air
     _FillValue: -32767
     valid_min: -30000
     valid_max: 25000
 path = /geophysical_data
 unlimited dimensions: 
 current shape = (1709, 1272, 172)
 filling on,
 'Rrs_unc': <class 'netCDF4.Variable'>
 int16 Rrs_unc(number_of_lines, pixels_per_line, wavelength_3d)
     long_name: Uncertainty in remote sensing reflectance
     scale_factor: 5e-08
     add_offset: 0.0015
     units: sr^-1
     coordinates: longitude latitude
     _FillValue: -32767
     valid_min: -30000
     valid_max: 30000
 path = /geophysical_data
 unlimited dimensions: 
 current shape = (1709, 1272, 172)
 filling on,
 'aot_865': <class 'netCDF4.Variabl